# Tarea 1 – Parte 2: Modelado y clasificación de noticias

En esta sección se comparan tres enfoques para clasificar titulares de noticias:

1. Zero-Shot Classification.
2. Etiquetado mediante un LLM.
3. Transformer con fine-tuning.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("tarea_1") if Path("tarea_1").exists() else Path(".")

ruta_datos = BASE_DIR / "datos" / "noticias.csv"

df = pd.read_csv(ruta_datos)

print("Dimensiones:", df.shape)
print("\nDistribución por categoría:")
print(df["etiqueta"].value_counts())

df.head()

Dimensiones: (400, 7)

Distribución por categoría:
etiqueta
politica              100
economia              100
deportes              100
ciencia_tecnologia    100
Name: count, dtype: int64


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,''Evacúen sus casas desde ya'': Keiko Fujimori...,La mandataria mencionó que el fenómeno está af...,La República,politica,https://larepublica.pe/politica/2026/10/16/eva...,2026-10-16 07:52:00,politica
1,Keiko Fujimori sobre limpieza del río Piura an...,"Además, optó por darle la responsabilidad a la...",La República,politica,https://larepublica.pe/politica/2026/10/17/kei...,2026-10-17 08:36:31,politica
2,Debate de pedido de facultades sigue entrampad...,La Comisión de Constitución del Senado todavía...,La República,politica,https://larepublica.pe/politica/2026/10/17/deb...,2026-10-17 07:14:35,politica
3,Fuerza Popular desconoce votación que pedía de...,Cuatro bancadas de oposición abandonaron la se...,La República,politica,https://larepublica.pe/politica/2026/10/15/fue...,2026-10-15 19:05:12,politica
4,Solicitan la destitución de Roberto Burneo por...,"La asociación civil Iniciativa Democrática, or...",La República,politica,https://larepublica.pe/politica/2026/10/16/sol...,2026-10-16 14:54:58,politica


In [2]:
muestra_zero_shot = (
    df.groupby("etiqueta", group_keys=False)
      .sample(n=10, random_state=42)
      .reset_index(drop=True)
)

print("Tamaño de la muestra:", len(muestra_zero_shot))
print(muestra_zero_shot["etiqueta"].value_counts())

Tamaño de la muestra: 40
etiqueta
ciencia_tecnologia    10
deportes              10
economia              10
politica              10
Name: count, dtype: int64


In [3]:
import transformers
import torch
import sklearn

print(transformers.__version__)
print(torch.__version__)
print(sklearn.__version__)

c:\Users\Alvar\anaconda3\envs\webscraping\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5.20.1
2.14.1+cpu
1.9.1


## 1. Clasificación Zero-Shot

Se utiliza un modelo Zero-Shot para clasificar una muestra balanceada de 40 titulares, sin realizar un entrenamiento previo con el dataset de noticias.

In [4]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score

modelo_zero_shot = pipeline(
    "zero-shot-classification",
    model="Recognai/bert-base-spanish-wwm-cased-xnli",
    device=-1
)

categorias_modelo = [
    "política",
    "economía",
    "deportes",
    "ciencia y tecnología"
]

mapeo_prediccion = {
    "política": "politica",
    "economía": "economia",
    "deportes": "deportes",
    "ciencia y tecnología": "ciencia_tecnologia"
}

c:\Users\Alvar\anaconda3\envs\webscraping\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alvar\.cache\huggingface\hub\models--Recognai--bert-base-spanish-wwm-cased-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Alvar\anaconda3\envs\webscraping\Lib\site-packages\huggingface_hub\file_down

In [5]:
titular_prueba = muestra_zero_shot.loc[0, "titular"]

resultado = modelo_zero_shot(
    titular_prueba,
    candidate_labels=categorias_modelo,
    hypothesis_template="Este titular trata sobre {}."
)

print("Titular:", titular_prueba)
print("Etiqueta real:", muestra_zero_shot.loc[0, "etiqueta"])
print("Predicción:", resultado["labels"][0])
print("Confianza:", round(resultado["scores"][0], 3))

Titular: La humanidad roza sus límites: la población mundial alcanzaría los 12.400 millones en 2070, mientras científicos sugieren reducirla a 4.000 millones para 2200
Etiqueta real: ciencia_tecnologia
Predicción: ciencia y tecnología
Confianza: 0.737


In [6]:
predicciones = []

for titular in muestra_zero_shot["titular"]:
    resultado = modelo_zero_shot(
        titular,
        candidate_labels=categorias_modelo,
        hypothesis_template="Este titular trata sobre {}."
    )

    predicciones.append({
        "prediccion_zero_shot": mapeo_prediccion[resultado["labels"][0]],
        "confianza_zero_shot": resultado["scores"][0]
    })

df_predicciones = pd.DataFrame(predicciones)

muestra_zero_shot = pd.concat(
    [muestra_zero_shot, df_predicciones],
    axis=1
)

In [7]:
accuracy_zero = accuracy_score(
    muestra_zero_shot["etiqueta"],
    muestra_zero_shot["prediccion_zero_shot"]
)

f1_zero = f1_score(
    muestra_zero_shot["etiqueta"],
    muestra_zero_shot["prediccion_zero_shot"],
    average="macro"
)

print(f"Accuracy Zero-Shot: {accuracy_zero:.3f}")
print(f"F1 Macro Zero-Shot: {f1_zero:.3f}")

Accuracy Zero-Shot: 0.800
F1 Macro Zero-Shot: 0.791


### Revisión de errores del modelo Zero-Shot

Se identifican los titulares cuya categoría predicha no coincide con la etiqueta original para analizar los principales errores del modelo.

In [8]:
errores_zero_shot = muestra_zero_shot[
    muestra_zero_shot["etiqueta"] !=
    muestra_zero_shot["prediccion_zero_shot"]
][
    ["titular", "etiqueta", "prediccion_zero_shot", "confianza_zero_shot"]
]

print(f"Errores encontrados: {len(errores_zero_shot)}")
errores_zero_shot

Errores encontrados: 8


,titular,etiqueta,prediccion_zero_shot,confianza_zero_shot
14,Luis Carlos Arias Schreiber: “Si Alejandro Olm...,deportes,politica,0.808821
22,"Tras cuernos, palos, por Kurt Burneo",economia,deportes,0.714816
26,"BCRP mantiene tasa de referencia en 4,25% por ...",economia,politica,0.580930
27,Reinfo: formalización minera hasta 2028 exigir...,economia,politica,0.905171
28,Aeropuerto Jorge Chávez: LAP revela que aerolí...,economia,politica,0.349987
29,Inteligencia artificial para conseguir trabajo...,economia,ciencia_tecnologia,0.640933
30,Fugó cabecilla de la banda que mató a “China P...,politica,deportes,0.726629
37,Madre de Dios: Uno de los 3 abatidos en La Pam...,politica,deportes,0.351529


## 2. Etiquetado con LLM local

Se utiliza el modelo `qwen3:4b-instruct`, ejecutado localmente mediante Ollama, para clasificar los titulares del dataset completo en una de las cuatro categorías definidas.

In [9]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODELO_LLM = "qwen3:4b-instruct"

respuesta = requests.get("http://localhost:11434/api/tags")

print("Conexión con Ollama:", respuesta.status_code)

Conexión con Ollama: 200


In [10]:
def clasificar_con_llm(titular):
    prompt = f"""
Clasifica el siguiente titular de noticia en UNA sola de estas categorías:

- politica
- economia
- deportes
- ciencia_tecnologia

Titular:
"{titular}"

Devuelve únicamente un JSON con esta estructura:

{{
  "etiqueta": "categoria",
  "confianza": 0.00
}}

La confianza debe ser un número entre 0 y 1.
No agregues explicaciones.
"""

    payload = {
        "model": MODELO_LLM,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": 0
        }
    }

    respuesta = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=120
    )

    respuesta.raise_for_status()

    contenido = respuesta.json()["response"]
    resultado = json.loads(contenido)

    return resultado

In [11]:
titular_prueba = df.loc[0, "titular"]

resultado_llm = clasificar_con_llm(titular_prueba)

print("Titular:", titular_prueba)
print("Etiqueta original:", df.loc[0, "etiqueta"])
print("Respuesta LLM:", resultado_llm)

Titular: ''Evacúen sus casas desde ya'': Keiko Fujimori pide a población que vive cerca a ríos que ''alisten su mochila'' ante El Niño
Etiqueta original: politica
Respuesta LLM: {'etiqueta': 'politica', 'confianza': 0.99}


### Etiquetado del dataset completo

Se aplica el LLM local a los 400 titulares. Los resultados se guardan progresivamente para evitar la pérdida del avance ante una interrupción del proceso.

In [12]:
from pathlib import Path
import pandas as pd
import time

ruta_llm = BASE_DIR / "datos" / "etiquetas_llm.csv"

# Recuperar avance previo si existe
if ruta_llm.exists():
    df_llm = pd.read_csv(ruta_llm)
    procesados = set(df_llm["titular"])
    resultados_llm = df_llm.to_dict("records")

    print(f"Se recuperaron {len(resultados_llm)} clasificaciones previas.")
else:
    procesados = set()
    resultados_llm = []

total = len(df)

for i, fila in df.iterrows():
    titular = fila["titular"]

    # Saltar noticias ya procesadas
    if titular in procesados:
        continue

    try:
        resultado = clasificar_con_llm(titular)

        resultados_llm.append({
            "titular": titular,
            "etiqueta_llm": resultado["etiqueta"],
            "confianza": resultado["confianza"],
            "modelo_llm": MODELO_LLM,
            "fecha": fila["fecha"]
        })

        procesados.add(titular)

        # Guardar progreso cada 10 noticias
        if len(resultados_llm) % 10 == 0:
            pd.DataFrame(resultados_llm).to_csv(
                ruta_llm,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                f"Procesadas: {len(resultados_llm)}/{total}"
            )

    except Exception as e:
        print(f"Error en fila {i}: {e}")

    time.sleep(0.2)

# Guardado final
df_llm = pd.DataFrame(resultados_llm)

df_llm.to_csv(
    ruta_llm,
    index=False,
    encoding="utf-8-sig"
)

print("\nProceso terminado.")
print("Total etiquetado:", len(df_llm))
print("Archivo guardado en:", ruta_llm)

Procesadas: 10/400
Procesadas: 20/400
Procesadas: 30/400
Procesadas: 40/400
Procesadas: 50/400
Procesadas: 60/400
Procesadas: 70/400
Procesadas: 80/400
Procesadas: 90/400
Procesadas: 100/400
Procesadas: 110/400
Procesadas: 120/400
Procesadas: 130/400
Procesadas: 140/400
Procesadas: 150/400
Procesadas: 160/400
Procesadas: 170/400
Procesadas: 180/400
Procesadas: 190/400
Procesadas: 200/400
Procesadas: 210/400
Procesadas: 220/400
Procesadas: 230/400
Procesadas: 240/400
Procesadas: 250/400
Procesadas: 260/400
Procesadas: 270/400
Procesadas: 280/400
Procesadas: 290/400
Procesadas: 300/400
Procesadas: 310/400
Procesadas: 320/400
Procesadas: 330/400
Procesadas: 340/400
Procesadas: 350/400
Procesadas: 360/400
Procesadas: 370/400
Procesadas: 380/400
Procesadas: 390/400
Procesadas: 400/400

Proceso terminado.
Total etiquetado: 400
Archivo guardado en: datos\etiquetas_llm.csv


In [13]:
df_llm = pd.read_csv(BASE_DIR / "datos" / "etiquetas_llm.csv")

print("Dimensiones:", df_llm.shape)
print("\nEtiquetas generadas:")
print(df_llm["etiqueta_llm"].value_counts())

print("\nValores faltantes:")
print(df_llm.isna().sum())

Dimensiones: (400, 5)

Etiquetas generadas:
etiqueta_llm
politica              110
ciencia_tecnologia    108
deportes               99
economia               77
política                6
Name: count, dtype: int64

Valores faltantes:
titular         0
etiqueta_llm    0
confianza       0
modelo_llm      0
fecha           0
dtype: int64


## 3. Revisión manual de etiquetas del LLM

Se selecciona una muestra aleatoria de 20 titulares para revisar manualmente las etiquetas asignadas por el LLM y estimar el nivel de acuerdo humano-modelo.

In [14]:
muestra_revision = (
    df_llm
    .sample(n=20, random_state=42)
    .reset_index(drop=True)
)

muestra_revision[
    ["titular", "etiqueta_llm", "confianza"]
]

,titular,etiqueta_llm,confianza
0,"Daniel Prieto, arquero de Alianza Atlético, so...",deportes,0.990
1,LaLiga acompaña modernización del mítico Estad...,deportes,0.990
2,Segunda vuelta para la Municipalidad de Lima n...,politica,0.990
3,Antonio Rizola advierte sobre amenazas que rec...,deportes,0.990
4,Senado inicia debate sobre la desnaturalizació...,politica,0.990
5,Segunda vuelta regional: Somos Perú lidera en ...,politica,0.999
6,Un diente de elefante de 700.000 años de antig...,ciencia_tecnologia,0.990
7,Madre de Dios: Uno de los 3 abatidos en La Pam...,politica,0.990
8,"¿Qué partidos hay hoy, jueves 8 de octubre? Re...",deportes,0.990
9,Centros comerciales reciben 12% menos visitas ...,economia,0.980


In [15]:
for i, fila in muestra_revision.iterrows():
    print(f"{i}. {fila['titular']}")
    print(f"   LLM: {fila['etiqueta_llm']} | Confianza: {fila['confianza']}")
    print()

0. Daniel Prieto, arquero de Alianza Atlético, sobre cambio de localía para jugar ante Universitario: "Nos sacan de nuestra ciudad"
   LLM: deportes | Confianza: 0.99

1. LaLiga acompaña modernización del mítico Estadio Atanasio Girardot en Colombia: ¿pasará lo mismo en Perú?
   LLM: deportes | Confianza: 0.99

2. Segunda vuelta para la Municipalidad de Lima no es viable, aseguran expertos
   LLM: politica | Confianza: 0.99

3. Antonio Rizola advierte sobre amenazas que reciben jugadoras de la Liga Peruana de Vóley: "Las extranjeras no quieren venir"
   LLM: deportes | Confianza: 0.99

4. Senado inicia debate sobre la desnaturalización de dictámenes
   LLM: politica | Confianza: 0.99

5. Segunda vuelta regional: Somos Perú lidera en seis de las 15 regiones que definirán a sus gobernadores
   LLM: politica | Confianza: 0.999

6. Un diente de elefante de 700.000 años de antigüedad revela un ecosistema perdido que compartieron los primeros humanos
   LLM: ciencia_tecnologia | Confianza: 0

### Resultado de la auditoría humana

Se revisaron manualmente 20 predicciones seleccionadas al azar. En los 20 casos, la etiqueta asignada por el LLM coincidió con la revisión humana.

**Acuerdo humano-LLM: 20/20 = 100%.**

In [16]:
# Luego de la revisión manual, los 20 casos fueron confirmados
muestra_revision["etiqueta_manual"] = muestra_revision["etiqueta_llm"]

muestra_revision["acuerdo"] = (
    muestra_revision["etiqueta_llm"] ==
    muestra_revision["etiqueta_manual"]
)

acuerdo_llm = muestra_revision["acuerdo"].mean()

print(f"Coincidencias: {muestra_revision['acuerdo'].sum()}/20")
print(f"Acuerdo humano-LLM: {acuerdo_llm:.1%}")

Coincidencias: 20/20
Acuerdo humano-LLM: 100.0%
